# Clusterização de risco em apostas — notebook completo

Do transacional Pix à régua de ação, em cinco etapas: gerar a base, identificar a casa,
construir as features, clusterizar e priorizar.

**Dados fictícios.** A base é gerada por simulação neste próprio notebook, com arquétipos
latentes de comportamento que servem de verdade de terreno. Nenhuma pessoa, conta ou casa
de apostas real foi usada.

**Base regulatória do método de rastreio:** Portaria Normativa SPA/MF nº 615/2024 —
depósito só de conta de mesma titularidade, prêmio de volta na mesma conta, e vedação a
espécie, boleto, criptoativo e cartão de crédito.

Semente fixa (`20260901`).

**Uma diferença esperada:** este notebook regenera a base e modela **num único fluxo
aleatório**, enquanto o relatório foi produzido por scripts modulares (geração, features,
clusterização) que consomem a semente em outra ordem. Os números batem na primeira casa e
divergem na terceira — por exemplo, a captura dos casos críticos sai 98,8% aqui contra
99,9% no relatório. As conclusões são as mesmas; o que muda é o sorteio.

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             adjusted_rand_score, roc_auc_score)
import matplotlib.pyplot as plt

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
SEED = 20260901
rng = np.random.default_rng(SEED)
print("pronto")

---
## 1. Gerar a base transacional

Cada cliente recebe um **arquétipo latente**. O parâmetro primário é o **comprometimento de
renda** — o ticket é derivado dele. Ancorar na grandeza de negócio evita gerar gente que
aposta vinte vezes a própria renda.

In [ ]:
N_CLIENTES = 10_000
INICIO, FIM = date(2026, 3, 1), date(2026, 8, 31)
N_DIAS = (FIM - INICIO).days + 1

MARCAS = ["AlfaBet", "BetOrion", "CanaBet", "DeltaAposta", "EloBet", "FenixBet",
          "GaleraBet", "HorizonBet", "IguacuBet", "JaguarBet", "KairosBet", "LunaBet"]
CASAS = pd.DataFrame({
    "id_casa": [f"C{i+1:02d}" for i in range(len(MARCAS))],
    "marca": MARCAS,
    "cnpj": [f"{rng.integers(10,99)}.{rng.integers(100,999)}.{rng.integers(100,999)}"
             f"/0001-{rng.integers(10,99)}" for _ in MARCAS],
    "dominio": [f"{m.lower()}.bet.br" for m in MARCAS]})
peso_casa = np.array([28,19,13,9,7,6,5,4,3,3,2,1], dtype=float); peso_casa /= peso_casa.sum()

# comp = comprometimento mensal da renda (mediana, sigma); freq = aportes/mes (log)
ARQ = {
 "Recreativo": dict(p=.55, comp=(.006,.60), freq=(0.70,.50), madru=.05, esc=1.00, casas=(1,2), ret=.72, sal=.10),
 "Regular":    dict(p=.25, comp=(.025,.55), freq=(1.90,.45), madru=.11, esc=1.02, casas=(1,3), ret=.70, sal=.20),
 "Intenso":    dict(p=.13, comp=(.090,.50), freq=(3.00,.40), madru=.24, esc=1.08, casas=(2,5), ret=.66, sal=.34),
 "Compulsivo": dict(p=.07, comp=(.280,.45), freq=(3.90,.40), madru=.42, esc=1.18, casas=(3,8), ret=.60, sal=.55)}
NOMES_ARQ = list(ARQ); P_ARQ = np.array([ARQ[a]["p"] for a in NOMES_ARQ])
print({a: ARQ[a]["p"] for a in NOMES_ARQ})

In [ ]:
arq = rng.choice(NOMES_ARQ, size=N_CLIENTES, p=P_ARQ)
renda = np.clip(np.exp(np.log(2600) + rng.normal(0, .62, N_CLIENTES)), 900, 60_000).round(2)

cli = pd.DataFrame({"id_cliente": [f"K{i+1:06d}" for i in range(N_CLIENTES)],
                    "arquetipo": arq, "renda_mensal": renda,
                    "dia_salario": rng.integers(1, 8, N_CLIENTES)})

lam, comp, pm, es, nc, pr_, ps = [], [], [], [], [], [], []
for a in arq:
    c = ARQ[a]
    lam.append(np.exp(rng.normal(*c["freq"])))
    comp.append(float(np.clip(np.exp(np.log(c["comp"][0]) + rng.normal(0, c["comp"][1])), .0005, 1.5)))
    pm.append(np.clip(rng.normal(c["madru"], .06), 0, .85))
    es.append(max(.85, rng.normal(c["esc"], .06)))
    nc.append(rng.integers(c["casas"][0], c["casas"][1] + 1))
    pr_.append(np.clip(rng.normal(c["ret"], .07), .30, .92))
    ps.append(np.clip(rng.normal(c["sal"], .08), 0, .90))

cli["lambda_mensal"] = lam; cli["comp_alvo"] = comp
# o ticket e DERIVADO do orcamento mensal, nao sorteado livremente
cli["ticket_base"] = (cli.renda_mensal * cli.comp_alvo / np.maximum(cli.lambda_mensal, .5)).clip(3, 20000)
cli["p_madrugada"] = pm; cli["escalada"] = es; cli["n_casas"] = nc
cli["p_retorno"] = pr_; cli["p_pos_salario"] = ps
cli.head(3)

In [ ]:
linhas = []
datas = [INICIO + timedelta(days=int(d)) for d in range(N_DIAS)]
mes_do_dia = np.array([(d.year-INICIO.year)*12 + d.month-INICIO.month for d in datas])
dia_do_mes = np.array([d.day for d in datas])

for i in range(N_CLIENTES):
    r = cli.iloc[i]
    casas = rng.choice(CASAS.id_casa.values, size=int(r.n_casas), replace=False, p=peso_casa)
    for m in range(6):
        fator = r.escalada ** m                 # so o VALOR escala; a frequencia fica estavel
        n_ap = rng.poisson(r.lambda_mensal)
        if n_ap == 0:
            continue
        dias = np.where(mes_do_dia == m)[0]
        pesos = np.ones(len(dias))
        d_m = dia_do_mes[dias]
        pesos[(d_m >= r.dia_salario) & (d_m <= r.dia_salario + 2)] += r.p_pos_salario * 12
        pesos /= pesos.sum()
        for d in rng.choice(dias, size=n_ap, p=pesos):
            valor = float(np.clip(rng.lognormal(np.log(r.ticket_base * fator), .55), 5, 25_000))
            hora = int(rng.integers(0, 6)) if rng.random() < r.p_madrugada else int(
                rng.choice(range(6, 24), p=np.array(
                    [1,2,3,4,5,6,7,7,6,6,7,8,9,9,8,6,4,2])/100))
            casa = str(rng.choice(casas))
            linhas.append((r.id_cliente, datas[d].isoformat(), hora, casa, "Aporte", round(valor, 2)))
            if rng.random() < r.p_retorno * .55:   # premio volta para a MESMA conta
                dv = datas[d] + timedelta(days=int(rng.integers(0, 3)))
                linhas.append((r.id_cliente, dv.isoformat(), int(rng.integers(0, 24)), casa,
                               "Premio", round(valor * float(np.clip(rng.lognormal(np.log(.9), .7), .05, 6)), 2)))

trx = pd.DataFrame(linhas, columns=["id_cliente","data","hora","id_casa","tipo","valor"])
trx = trx[trx.data <= FIM.isoformat()].merge(CASAS[["id_casa","marca","cnpj"]], on="id_casa")
trx["data"] = pd.to_datetime(trx["data"])
print(f"{len(trx):,} transacoes  ({(trx.tipo=='Aporte').sum():,} aportes, "
      f"{(trx.tipo=='Premio').sum():,} premios)")

---
## 2. Identificar a casa de apostas

Na base sintética o vínculo já vem pronto. Em produção, é este o passo — e ele é
determinístico porque o CNPJ do recebedor vem na mensagem do Pix e a lista da SPA é pública.

In [ ]:
# CAMADA 1 — CNPJ contra a lista oficial (deterministica)
def identifica_por_cnpj(pix, casas):
    p = pix.copy(); c = casas.copy()
    p["cnpj_norm"] = p["cnpj"].str.replace(r"\D", "", regex=True)
    c["cnpj_norm"] = c["cnpj"].str.replace(r"\D", "", regex=True)
    return p.merge(c[["cnpj_norm", "marca", "dominio"]], on="cnpj_norm",
                   how="left", suffixes=("", "_casa"))

# CAMADA 2 — nome do favorecido, para quando ha subadquirente no meio
def normaliza(nome):
    import unicodedata, re
    s = unicodedata.normalize("NFKD", str(nome)).encode("ascii", "ignore").decode().upper()
    return re.sub(r"\b(LTDA|SA|S/A|ME|EIRELI|EMPREENDIMENTOS)\b", "", s).strip()

ident = identifica_por_cnpj(trx, CASAS)
print(f"identificadas pela camada 1: {ident.marca_casa.notna().mean():.1%}")

---
## 3. As features: quatro blocos

Frequência, valor, comprometimento e padrão. As três que merecem atenção: **perda líquida**
(aportado − recebido), **escalada** (último bimestre / primeiro) e **máximo de aportes num
dia** (perseguição de perda).

In [ ]:
JANELA = 6
ap = trx[trx.tipo == "Aporte"].copy(); pr = trx[trx.tipo == "Premio"]
ap["mes"] = ap.data.dt.to_period("M"); ref = trx.data.max()

f = ap.groupby("id_cliente").agg(n_aportes=("valor","size"), dias_distintos=("data","nunique"),
                                 ultimo=("data","max")).reset_index()
f["freq_mensal"] = f.n_aportes / JANELA
f["recencia_dias"] = (ref - f.ultimo).dt.days
por_dia = ap.groupby(["id_cliente","data"]).size().reset_index(name="n")
f = f.merge(por_dia.groupby("id_cliente")["n"].max().rename("max_aportes_dia").reset_index(), on="id_cliente")

v = ap.groupby("id_cliente").agg(valor_total=("valor","sum"), ticket_medio=("valor","mean"),
                                 maior_aporte=("valor","max")).reset_index()
v["gasto_mensal"] = v.valor_total / JANELA
v = v.merge(pr.groupby("id_cliente")["valor"].sum().rename("premio_total").reset_index(),
            on="id_cliente", how="left").fillna({"premio_total": 0.0})
v["perda_liquida"] = v.valor_total - v.premio_total          # o que saiu do bolso de fato
v["perda_mensal"] = v.perda_liquida / JANELA

ap["madrugada"] = ap.hora.between(0, 5)
pat = ap.groupby("id_cliente").agg(pct_madrugada=("madrugada","mean"),
                                   n_casas=("id_casa","nunique")).reset_index()

meses = sorted(ap.mes.unique()); prim, ult = meses[:2], meses[-2:]
bim = (ap.assign(b=np.where(ap.mes.isin(prim), "ini", np.where(ap.mes.isin(ult), "fim", "meio")))
       .query("b != 'meio'").groupby(["id_cliente","b"])["valor"].sum().unstack(fill_value=0.))
bim = bim.reindex(columns=["ini","fim"], fill_value=0.)
bim["escalada"] = (bim.fim + 1) / (bim.ini + 1)              # o +1 evita divisao por zero
pat = pat.merge(bim[["escalada"]].reset_index(), on="id_cliente", how="left")
pat["escalada"] = pat.escalada.fillna(1.0)

ap = ap.merge(cli[["id_cliente","dia_salario"]], on="id_cliente")
ap["pos_sal"] = ap.data.dt.day.between(ap.dia_salario, ap.dia_salario + 3)
pat = pat.merge(ap.groupby("id_cliente")["pos_sal"].mean().rename("pct_pos_salario").reset_index(),
                on="id_cliente")

df = cli[["id_cliente","arquetipo","renda_mensal"]].merge(f, on="id_cliente").merge(
     v, on="id_cliente").merge(pat, on="id_cliente")
df["comprometimento"] = df.gasto_mensal / df.renda_mensal
print(f"{len(df):,} clientes com aporte")
df.groupby("arquetipo")[["freq_mensal","ticket_medio","gasto_mensal","comprometimento",
                         "pct_madrugada","escalada","n_casas"]].median().reindex(
    ["Recreativo","Regular","Intenso","Compulsivo"]).round(4)

---
## 4. Preparar: log e padronização

K-Means usa distância euclidiana. Sem log, os extremos definem os centroides. Sem
padronização, a frequência sozinha manda na distância.

In [ ]:
EIXOS = ["freq_mensal", "ticket_medio", "comprometimento"]
SINAIS = ["pct_madrugada", "escalada", "n_casas", "max_aportes_dia", "pct_pos_salario"]
TODAS = EIXOS + ["pct_madrugada", "escalada", "n_casas", "pct_pos_salario"]
LOG1P = {"freq_mensal","ticket_medio","n_casas","max_aportes_dia"}
LOG = {"comprometimento","escalada"}

def transforma(X, cols):
    X = X.copy()
    for c in cols:
        if c in LOG1P: X[c] = np.log1p(X[c])
        elif c in LOG: X[c] = np.log(X[c].clip(lower=1e-4))
    return X

def prepara(cols):
    Xt = transforma(df[cols], cols)
    sc = StandardScaler().fit(Xt)
    return sc.transform(Xt), sc

X3, sc3 = prepara(EIXOS)
X7, _ = prepara(TODAS)

pd.DataFrame({"feature": TODAS,
              "antes": [df[c].skew() for c in TODAS],
              "depois": [(np.log1p(df[c]) if c in LOG1P else
                          np.log(df[c].clip(lower=1e-4)) if c in LOG else df[c]).skew()
                         for c in TODAS]}).round(2)

---
## 5. Quantos grupos?

Os critérios discordam — e a discordância informa. Silhueta prefere k=2; Davies-Bouldin,
k=4. A estrutura mais forte é binária; os quatro grupos são granularidade operacional.

In [ ]:
ORDEM_ARQ = ["Recreativo","Regular","Intenso","Compulsivo"]
arq_cod = df.arquetipo.map({a:i for i,a in enumerate(ORDEM_ARQ)}).values

def varredura(X, ks=range(2, 9), amostra=4000):
    idx = rng.choice(len(X), size=min(amostra, len(X)), replace=False)
    return pd.DataFrame([{
        "k": k, "inercia": (km := KMeans(k, n_init=20, random_state=SEED).fit(X)).inertia_,
        "silhueta": silhouette_score(X[idx], km.labels_[idx]),
        "davies_bouldin": davies_bouldin_score(X, km.labels_),
        "ari_arquetipo": adjusted_rand_score(arq_cod, km.labels_)} for k in ks])

var3, var7 = varredura(X3), varredura(X7)
print("3 EIXOS DE VOLUME"); print(var3.round(4).to_string(index=False))
K = 4

---
## 6. Mais variáveis deu clusteres piores

A hipótese natural — jogar tudo no K-Means — foi negada pelos dados, em todos os critérios.

In [ ]:
def compara(X, k):
    km = KMeans(k, n_init=20, random_state=SEED).fit_predict(X)
    gm = GaussianMixture(k, covariance_type="full", n_init=5, random_state=SEED).fit_predict(X)
    idx = rng.choice(len(X), 4000, replace=False)     # Ward e O(n^2)
    wd = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X[idx])
    return adjusted_rand_score(km, gm), adjusted_rand_score(km[idx], wd)

def estabilidade(X, k, n=30, frac=.80):
    base = KMeans(k, n_init=20, random_state=SEED).fit(X)
    a = [adjusted_rand_score(base.labels_[i],
         KMeans(k, n_init=10, random_state=SEED+b+1).fit_predict(X[i]))
         for b in range(n) for i in [rng.choice(len(X), int(frac*len(X)), replace=False)]]
    return base, np.array(a)

base3, e3 = estabilidade(X3, K); base7, e7 = estabilidade(X7, K)
g3, w3 = compara(X3, K); g7, w7 = compara(X7, K)

pd.DataFrame({
    "criterio": ["Silhueta", "Davies-Bouldin (menor melhor)", "Estabilidade (ARI medio)",
                 "Estabilidade (pior caso)", "KMeans x GMM", "KMeans x Ward",
                 "Recuperacao do arquetipo"],
    "3 eixos": [var3.loc[var3.k==K,"silhueta"].iloc[0], var3.loc[var3.k==K,"davies_bouldin"].iloc[0],
                e3.mean(), e3.min(), g3, w3, adjusted_rand_score(arq_cod, base3.labels_)],
    "7 features": [var7.loc[var7.k==K,"silhueta"].iloc[0], var7.loc[var7.k==K,"davies_bouldin"].iloc[0],
                   e7.mean(), e7.min(), g7, w7, adjusted_rand_score(arq_cod, base7.labels_)]}).round(4)

---
## 7. Os quatro grupos

In [ ]:
med = pd.DataFrame({"c": base3.labels_, "x": df.comprometimento}).groupby("c")["x"].median()
mapa = {c: i+1 for i, c in enumerate(med.sort_values().index)}
df["cluster"] = [mapa[c] for c in base3.labels_]
NOMES = {1:"Recreativo", 2:"Ticket alto ocasional", 3:"Habitual", 4:"Intensivo"}
df["grupo"] = df.cluster.map(NOMES)

g = df.groupby("cluster")
perfil = g[["freq_mensal","ticket_medio","gasto_mensal","comprometimento","perda_mensal",
            "pct_madrugada","escalada","n_casas","max_aportes_dia","renda_mensal"]].median()
perfil.insert(0, "clientes", g.size())
perfil.insert(1, "pct_base", g.size()/len(df))
perfil["pct_perda_total"] = g.perda_liquida.sum() / df.perda_liquida.sum()
perfil.insert(0, "grupo", perfil.index.map(NOMES))
perfil.round(4)

In [ ]:
print("CLUSTER x ARQUETIPO (% da linha) — o modelo nunca viu essa coluna")
print((pd.crosstab(df.cluster, df.arquetipo, normalize="index")*100)[ORDEM_ARQ].round(1))
n4 = ((df.cluster==4) & (df.arquetipo=="Compulsivo")).sum()
print(f"\ncluster 4 contem {n4} dos {(df.arquetipo=='Compulsivo').sum()} casos criticos "
      f"({n4/(df.arquetipo=='Compulsivo').sum():.1%}) em {(df.cluster==4).mean():.1%} da base")
print(f"e concentra {perfil.loc[4,'pct_perda_total']:.1%} da perda liquida total")

---
## 8. Etapa 2: priorizar dentro do grupo de risco

Os sinais de padrão pioraram a clusterização. Aqui, como critério de **ordenação dentro da
fila**, eles funcionam.

In [ ]:
alvo = df.cluster == K
top = df[alvo].copy()
z = top[SINAIS].copy()
z["escalada"] = np.log(z.escalada.clip(lower=1e-4))
for c in ("n_casas","max_aportes_dia"): z[c] = np.log1p(z[c])
top["indice_alerta"] = ((z - z.mean()) / z.std()).mean(axis=1)

y = (top.arquetipo == "Compulsivo").astype(int)
print(f"AUC do indice: {roc_auc_score(y, top.indice_alerta):.4f}")
print(pd.DataFrame([{"sinal": s, "auc": roc_auc_score(
    y, np.log(top[s].clip(lower=1e-4)) if s=="escalada" else top[s])}
    for s in SINAIS + ["comprometimento"]]).sort_values("auc", ascending=False).round(4).to_string(index=False))

In [ ]:
t = top.sort_values("indice_alerta", ascending=False)
print(pd.DataFrame([{
    "pct_do_cluster": p, "clientes": (n := int(len(t)*p)),
    "pct_critico": (t.head(n).arquetipo=="Compulsivo").mean(),
    "captura": (t.head(n).arquetipo=="Compulsivo").sum()/y.sum()}
    for p in (.10,.20,.30,.40,.50,.75,1.0)]).round(4).to_string(index=False))

**Ressalva obrigatória:** esse AUC é alto porque os arquétipos sintéticos diferem
exatamente nesses sinais. Em dado real a separação seria bem menor. **O que transfere é o
método** — clusterizar por volume, ordenar por padrão — não o número.

In [ ]:
df["prioridade"] = "Sem alerta"
df.loc[alvo, "prioridade"] = pd.qcut(top.indice_alerta, [0,.5,.8,1],
                                     labels=["Monitorar","Atencao","Acao imediata"]).astype(str).values
df[alvo].groupby("prioridade").agg(
    clientes=("id_cliente","size"), comprometimento=("comprometimento","median"),
    perda_mensal=("perda_mensal","median"), madrugada=("pct_madrugada","median"),
    escalada=("escalada","median"),
    pct_critico=("arquetipo", lambda s: (s=="Compulsivo").mean())
).reindex(["Monitorar","Atencao","Acao imediata"]).round(4)

---
## 9. Os gráficos

In [ ]:
PAL = {"azul":"#2a78d6","laranja":"#eb6834","cinza":"#898781","grade":"#e1e0d9","fundo":"#fcfcfb"}
plt.rcParams.update({"figure.facecolor":PAL["fundo"], "axes.facecolor":PAL["fundo"],
                     "axes.spines.top":False, "axes.spines.right":False, "legend.frameon":False,
                     "grid.color":PAL["grade"], "xtick.color":PAL["cinza"], "ytick.color":PAL["cinza"]})

fig, (a, b) = plt.subplots(1, 2, figsize=(12.5, 4.3))
x = np.arange(len(perfil)); w = .36
a.bar(x-w/2, perfil.pct_base, w, color=PAL["azul"], label="% dos clientes")
a.bar(x+w/2, perfil.pct_perda_total, w, color=PAL["laranja"], label="% da perda liquida")
a.set_xticks(x); a.set_xticklabels(perfil.grupo, fontsize=9)
a.legend(); a.grid(True, axis="y"); a.set_axisbelow(True)
a.set_title("Onde esta o dinheiro perdido", loc="left", fontweight="bold")

dados = [df.loc[df.cluster==c, "comprometimento"].clip(upper=1.2)*100 for c in sorted(df.cluster.unique())]
bp = b.boxplot(dados, vert=False, patch_artist=True, widths=.6,
               medianprops=dict(color=PAL["fundo"], lw=2))
for i, cx in enumerate(bp["boxes"]):
    cx.set_facecolor(PAL["laranja"] if i==3 else PAL["azul"]); cx.set_edgecolor(PAL["fundo"])
b.set_yticks(range(1,5)); b.set_yticklabels([NOMES[c] for c in sorted(df.cluster.unique())], fontsize=9)
b.axvline(10, color=PAL["cinza"], ls="--", lw=1.2)
b.set_xlabel("Comprometimento da renda (%)"); b.grid(True, axis="x"); b.set_axisbelow(True)
b.set_title("Comprometimento por grupo", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
pca = PCA(n_components=2, random_state=SEED).fit(X3)
proj = pca.transform(X3)
df["pc1"], df["pc2"] = proj[:,0], proj[:,1]
am = df.sample(4000, random_state=7)

fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.6))
for ax, c in zip(axes, sorted(df.cluster.unique())):
    ax.scatter(am.pc1, am.pc2, s=6, color=PAL["cinza"], alpha=.18, edgecolors="none")
    sel = am[am.cluster == c]
    ax.scatter(sel.pc1, sel.pc2, s=7, color=PAL["laranja"] if c==4 else PAL["azul"],
               alpha=.55, edgecolors="none")
    ax.set_title(f"{NOMES[c]} ({(df.cluster==c).sum():,})", loc="left", fontsize=10, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ("left","bottom"): ax.spines[s].set_visible(False)
plt.suptitle(f"Os grupos no plano principal — {pca.explained_variance_ratio_.sum():.0%} da variancia",
             x=.005, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

---
## 10. Exportar

In [ ]:
COLS = ["id_cliente","arquetipo","renda_mensal","freq_mensal","ticket_medio","gasto_mensal",
        "comprometimento","perda_liquida","perda_mensal","pct_madrugada","escalada",
        "n_casas","max_aportes_dia","pct_pos_salario","cluster","grupo","prioridade"]
df[COLS].to_csv("bets_clusters.csv", index=False)
CASAS.to_csv("casas_autorizadas.csv", index=False)
trx[trx.id_cliente.isin(df.id_cliente.sample(500, random_state=7))].to_csv(
    "bets_transacoes_amostra.csv", index=False)

# parametros para reaplicar o modelo fora do Python
pd.DataFrame({"feature": EIXOS, "media": sc3.mean_, "desvio": sc3.scale_}).to_csv(
    "parametros_escala.csv", index=False)
cent = pd.DataFrame(KMeans(K, n_init=20, random_state=SEED).fit(X3).cluster_centers_, columns=EIXOS)
cent["cluster"] = cent.index.map(mapa)
cent.sort_values("cluster").to_csv("parametros_centroides.csv", index=False)
print("arquivos gravados")

---
## Limitações

1. **Dados fictícios.** As correlações são as que impus; o que se demonstra é o método.
2. **Quatro grupos é escolha, não descoberta.** A estrutura mais forte é binária.
3. **Intenso e Compulsivo formam um contínuo.** O cluster tria; o índice ordena. Aumentar k
   fragmenta e piora a captura.
4. **Renda pode estar errada.** Todo o eixo de comprometimento depende dela. Para autônomo é
   estimada, e o erro contamina a variável mais importante. Segmente por qualidade da renda.
5. **O extrato vê a conta, não a pessoa.** Se aposta por outra instituição, você não vê. O
   comprometimento é um **piso**.
6. **Não é diagnóstico.** Jogo patológico tem critério clínico e profissional habilitado.
   O que sai daqui é fila de prioridade para proteção.
7. **Uso indefensável:** negar crédito, piorar taxa ou reduzir limite com base nestes grupos.